# Load Pre-processed Data
## Load từ data/gold/All_Beauty

Dùng data đã được xử lý sẵn từ lần chạy trước

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# === SETUP PATHS ===
PROJECT_DIR = Path(".")
# .resolve() giúp in ra đường dẫn tuyệt đối thực tế để bạn quan sát
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"\nFiles available:")

GOLD_DIR = PROJECT_DIR / "data" / "gold"
CATEGORY = "All_Beauty"
DATA_DIR = GOLD_DIR / CATEGORY

print(f"Data directory: {DATA_DIR}")
print(f"\nFiles available:")
for file in sorted(DATA_DIR.iterdir()):
    if file.is_file():
        size = file.stat().st_size / (1024 * 1024)  # MB
        print(f"  - {file.name:<40} ({size:.2f} MB)")

Data directory: /Users/mac/Downloads/MXH FINAL/data/gold/All_Beauty

Files available:
                                    (0.00 MB)
  - edges_review_item.parquet                (28.27 MB)
  - edges_review_review.parquet              (16.77 MB)
  - edges_user_item.parquet                  (25.53 MB)
  - edges_user_review.parquet                (42.32 MB)
  - nodes_item.parquet                       (2.34 MB)
  - nodes_review.parquet                     (203.87 MB)
  - nodes_user.parquet                       (17.77 MB)
  - pipeline_summary.json                    (0.00 MB)
  - review_features.parquet                  (36.36 MB)
  - review_splits.parquet                    (20.37 MB)
  - review_text_embeddings_All_Beauty.npy    (944.37 MB)
  - reviews_clean.parquet                    (393.99 MB)


## Load All Data

In [3]:
# === LOAD CLEANED REVIEWS ===
reviews_clean = pd.read_parquet(DATA_DIR / "reviews_clean.parquet")
print(f"Reviews: {len(reviews_clean)} rows")
print(f"Columns: {reviews_clean.columns.tolist()}")
print(reviews_clean.head())

Reviews: 644689 rows
Columns: ['review_id', 'user_id', 'item_id', 'rating', 'review_title', 'review_text', 'timestamp', 'helpful_vote', 'verified_purchase', 'text_raw', 'text_clean', 'text_length_chars', 'text_length_words', 'num_exclamation', 'num_question', 'num_uppercase_words', 'uppercase_word_ratio', 'is_short_review', 'is_very_short_review', 'text_fingerprint', 'duplicate_text_count', 'user_review_count', 'user_item_count', 'user_avg_rating', 'user_rating_std', 'user_first_review_time', 'user_last_review_time', 'user_verified_ratio', 'user_avg_text_length', 'user_active_days', 'user_reviews_per_day', 'user_singleton', 'item_review_count', 'item_user_count', 'item_avg_rating', 'item_rating_std', 'item_first_review_time', 'item_last_review_time', 'item_verified_ratio', 'item_avg_text_length', 'item_active_days', 'item_reviews_per_day', 'review_date', 'review_year', 'review_month', 'review_year_month', 'review_dayofweek', 'user_day_review_count', 'item_day_review_count', 'item_month

In [4]:
# === LOAD NODES ===
nodes_user = pd.read_parquet(DATA_DIR / "nodes_user.parquet")
nodes_review = pd.read_parquet(DATA_DIR / "nodes_review.parquet")
nodes_item = pd.read_parquet(DATA_DIR / "nodes_item.parquet")

print(f"Users: {len(nodes_user)}")
print(f"Reviews: {len(nodes_review)}")
print(f"Items: {len(nodes_item)}")

Users: 588332
Reviews: 644689
Items: 108784


In [5]:
# === LOAD EDGES ===
edges_user_review = pd.read_parquet(DATA_DIR / "edges_user_review.parquet")
edges_review_item = pd.read_parquet(DATA_DIR / "edges_review_item.parquet")
edges_user_item = pd.read_parquet(DATA_DIR / "edges_user_item.parquet")
edges_review_review = pd.read_parquet(DATA_DIR / "edges_review_review.parquet")

print(f"User-Review edges: {len(edges_user_review)}")
print(f"Review-Item edges: {len(edges_review_item)}")
print(f"User-Item edges: {len(edges_user_item)}")
print(f"Review-Review edges: {len(edges_review_review)}")

User-Review edges: 644689
Review-Item edges: 644689
User-Item edges: 644689
Review-Review edges: 267837


In [6]:
# === LOAD FEATURES ===
review_features = pd.read_parquet(DATA_DIR / "review_features.parquet")
review_splits = pd.read_parquet(DATA_DIR / "review_splits.parquet")

print(f"Review features: {review_features.shape}")
print(f"Review splits: {review_splits.shape}")
print(f"\nSplits:")
print(review_splits["split"].value_counts())

Review features: (644689, 39)
Review splits: (644689, 4)

Splits:
split
train    451282
test      96704
val       96703
Name: count, dtype: int64


In [7]:
# === LOAD EMBEDDINGS ===
embedding_path = DATA_DIR / f"review_text_embeddings_{CATEGORY}.npy"
if embedding_path.exists():
    review_embeddings = np.load(embedding_path)
    print(f"Embeddings shape: {review_embeddings.shape}")
else:
    print("No embeddings file found")

Embeddings shape: (644689, 384)


In [8]:
# === LOAD SUMMARY ===
with open(DATA_DIR / "pipeline_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)
print(f"Category: {summary['category']}")
print(f"Timestamp: {summary['timestamp']}")
print(f"\nStatistics:")
print(f"  Reviews: {summary['num_reviews']:,}")
print(f"  Users: {summary['num_users']:,}")
print(f"  Items: {summary['num_items']:,}")
print(f"  Total Edges: {sum(summary[k] for k in ['num_edges_user_review', 'num_edges_review_item', 'num_edges_user_item', 'num_edges_review_review']):,}")
print(f"\nLabel Distribution:")
for k, v in summary['weak_label_distribution'].items():
    pct = v / summary['num_reviews'] * 100
    print(f"  Weak label {k}: {v:,} ({pct:.1f}%)")
print(f"\nSplit Distribution:")
for k, v in summary['split_distribution'].items():
    pct = v / summary['num_reviews'] * 100
    print(f"  {k}: {v:,} ({pct:.1f}%)")
print("="*60)


PIPELINE SUMMARY
Category: All_Beauty


KeyError: 'timestamp'

## Sample Data Exploration

In [ ]:
# === EXPLORE REVIEWS ===
print("Sample reviews:")
print(nodes_review[["review_id", "user_id", "item_id", "rating", "weak_label", "split"]].head(10))

In [ ]:
# === EXPLORE USERS ===
print("Sample users:")
print(nodes_user[["user_id", "user_review_count", "user_item_count", "user_avg_rating"]].head(10))

In [ ]:
# === EXPLORE FEATURES ===
print("Review features statistics:")
print(review_features.describe())

In [ ]:
# === WEAK LABELS DISTRIBUTION ===
print("Weak label distribution:")
print(nodes_review["weak_label"].value_counts())
print(f"\nPercentage:")
print(nodes_review["weak_label"].value_counts(normalize=True) * 100)

## Ready to Use!

Tất cả data đã được load sẵn. Bạn có thể dùng:
- `reviews_clean` - Cleaned reviews
- `nodes_user`, `nodes_review`, `nodes_item` - Graph nodes
- `edges_*` - Graph edges
- `review_features` - Feature matrix
- `review_embeddings` - Text embeddings
- `summary` - Pipeline metadata